# 11 · Async: Waiting for Things & Fetching Data

Some things take time: a timer, a file, a server on the other side of the world. Today you learn how programs wait without freezing, and you fetch real data from the internet.

**By the end of this lesson you can:**
- Explain why JavaScript doesn't block while it waits
- Predict the order of output with `setTimeout`
- Use Promises with `async` / `await`, and `Promise.all`
- Fetch JSON from a web API and handle errors
- Recognise the same ideas in Python's `asyncio`

In [ ]:
# ▶️ Run this cell first, every time you open the notebook.
# It lets %%node cells run JavaScript and gives you check() in both languages.
%load_ext course_helpers

## 🔁 Warm-up
Answer from memory first: **no scrolling back, no notes**. Getting it wrong is fine. Trying to remember is what makes it stick.

1. What's the difference between a class and an instance?
2. Write a JavaScript class `Point` with `x`, `y` and a method `distanceToOrigin()`.
3. What does `super(...)` do?
4. What is a pull request?

**My answers** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️
4. ✏️

## Part 1: Some things take time
So far everything has happened instantly. But programs often have to wait. Should the program
**freeze** while it waits? A web page that froze every time it loaded an image would be awful.
JavaScript was made for the browser, so it never just waits. It says *"call me back when it's done"* and
keeps going.

Think of a café: you order, get a buzzer and sit down. You don't stand frozen at the counter.
When the buzzer goes off, you collect your coffee.

In [ ]:
%%node
// 🔮 In which order are the lines printed?
console.log("1. Order coffee");

setTimeout(() => {
  console.log("2. Coffee is ready ☕");
}, 1000);

console.log("3. Sit down and read the news");

🔍 `setTimeout(callback, ms)` means *"run this callback after `ms` milliseconds"*. JavaScript doesn't wait.
It registers the callback and goes straight on to line 3. One second later, the callback runs.
(Did you notice the last line appearing a moment later?)

In [ ]:
%%node
// 🔮 And with 0 milliseconds?
console.log("A");
setTimeout(() => console.log("B"), 0);
console.log("C");

🔍 Callbacks always wait until the code that's running now has completely finished. JavaScript does one
thing at a time and keeps a queue of callbacks to run when it's free. This is the **event loop**.

Python normally just waits (**blocks**):

In [ ]:
import time

print("1. Order coffee")
time.sleep(1)    # the whole program waits here
print("2. Coffee is ready ☕")
print("3. Sit down and read the news")

## Part 2: Promises and `async` / `await`
Callbacks inside callbacks get messy fast. Modern JavaScript uses **Promises**: an object that stands
for a value that will arrive **later** (the buzzer). `await` pauses *this piece of code* until the
promise is ready, without freezing everything else.

In [ ]:
%%node
// A function that returns a promise, fulfilled after some seconds
function brew(drink, seconds) {
  return new Promise((resolve) => {
    setTimeout(() => resolve(`${drink} is ready`), seconds * 1000);
  });
}

console.log("Ordering...");
const result = await brew("Coffee", 1);
console.log(result);

In our `%%node` cells you can use `await` anywhere. In normal code, `await` must be inside a function
marked `async`:

In [ ]:
%%node
function brew(drink, seconds) {
  return new Promise((resolve) => {
    setTimeout(() => resolve(`${drink} is ready`), seconds * 1000);
  });
}

async function breakfast() {
  const coffee = await brew("Coffee", 1);
  console.log(coffee);
  const tea = await brew("Tea", 1);
  console.log(tea);
  return "Breakfast done";
}

const start = Date.now();
const message = await breakfast();
console.log(message, `after ${((Date.now() - start) / 1000).toFixed(1)} s`);

## Part 3: Waiting for several things at once

In [ ]:
%%node
// 🔮 How many seconds will this take?
function brew(drink, seconds) {
  return new Promise((resolve) => {
    setTimeout(() => resolve(`${drink} is ready`), seconds * 1000);
  });
}

const start = Date.now();
const [coffee, tea] = await Promise.all([brew("Coffee", 1), brew("Tea", 1)]);
console.log(coffee, "/", tea);
console.log(`after ${((Date.now() - start) / 1000).toFixed(1)} s`);

🔍 `Promise.all` starts all of them **at the same time** and waits until all are done: 1 second instead
of 2. It's like ordering the coffee and the tea together.

Python has the same idea in its `asyncio` module, with almost the same words:

In [ ]:
import asyncio
import time

async def brew(drink, seconds):
    await asyncio.sleep(seconds)
    return f"{drink} is ready"

start = time.time()
coffee, tea = await asyncio.gather(brew("Coffee", 1), brew("Tea", 1))
print(coffee, "/", tea)
print(f"after {time.time() - start:.1f} s")

Top-level `await` in Python only works in notebooks. In a `.py` file you'd write `asyncio.run(main())`.
`async`, `await` and "wait for all" also exist in C#, Rust, Kotlin and Swift.

## Part 4: Fetching data from the internet
An **API** is a way for programs to talk to other programs, often over the web. You send a request to
a URL, and the server answers with data, usually as JSON (lesson 06).
The [PokéAPI](https://pokeapi.co) is free and needs no account. Open
https://pokeapi.co/api/v2/pokemon/pikachu in your browser to see the raw JSON.

In [ ]:
%%node --timeout 30
const response = await fetch("https://pokeapi.co/api/v2/pokemon/pikachu");
console.log("Status:", response.status);

const data = await response.json();
console.log(data.name);
console.log("Height:", data.height, "Weight:", data.weight);
console.log("Types:", data.types.map((t) => t.type.name));

🔍 Two `await`s: the first waits for the response to arrive, the second waits for its body to be read
and turned from JSON text into data. Status `200` means OK, `404` means "not found".

In [ ]:
import json
import urllib.request

# Many servers refuse requests that don't say which program is asking,
# so we send a "User-Agent" header, just like a browser does.
request = urllib.request.Request(
    "https://pokeapi.co/api/v2/pokemon/pikachu",
    headers={"User-Agent": "programming-fundamentals-course"},
)
with urllib.request.urlopen(request) as response:
    print("Status:", response.status)
    data = json.loads(response.read())

print(data["name"])
print("Height:", data["height"], "Weight:", data["weight"])
print("Types:", [t["type"]["name"] for t in data["types"]])

Python's built-in `urllib` blocks while it waits, and without the `User-Agent` header this server answers
`403 Forbidden`. Most Python programmers use the `requests` package instead (`pip install requests`):
`requests.get(url).json()`.

### When things go wrong

In [ ]:
%%node --timeout 30
async function getJson(url) {
  const response = await fetch(url);
  if (!response.ok) {
    throw new Error(`Request failed with status ${response.status}`);
  }
  return response.json();
}

try {
  const data = await getJson("https://pokeapi.co/api/v2/pokemon/not-a-pokemon");
  console.log(data.name);
} catch (err) {
  console.log("⚠️", err.message);
}

🔍 `fetch` only throws when the **network** fails. A 404 is a successful request with a sad answer, so
you have to check `response.ok` yourself.

## 🛠️ Make
### 1. `getPokemon`

In [ ]:
%%node --timeout 30
async function getPokemon(name) {
  // ✏️ Fetch https://pokeapi.co/api/v2/pokemon/<name>
  //    If the response is not ok, throw an Error with the message `No Pokémon called <name>`
  //    Otherwise return an object { name, weight, types }, where types is an array of type names
}

const bulbasaur = await getPokemon("bulbasaur");
check(bulbasaur.name, "bulbasaur");
check(bulbasaur.types, ["grass", "poison"]);

try {
  await getPokemon("pikachuuu");
  console.log("❌ should have thrown an error");
} catch (err) {
  check(err.message, "No Pokémon called pikachuuu");
}

### 2. The heaviest, fetched in parallel

In [ ]:
%%node --timeout 30
// ✏️ Paste your getPokemon function here


// ✏️ Fetch all three AT THE SAME TIME (Promise.all + map),
//    then find the name of the heaviest one (reduce, or a loop)
const names = ["pikachu", "snorlax", "charmander"];
let heaviest = null;

check(heaviest, "snorlax");

### 3. Countdown
No internet needed for this one.

In [ ]:
%%node
function wait(seconds) {
  return new Promise((resolve) => setTimeout(resolve, seconds * 1000));
}

// ✏️ Print 3, 2, 1 with one second between each, then "Go! 🚀"

### 4. `get_pokemon` in Python

In [ ]:
import json
import urllib.error
import urllib.request

def get_pokemon(name):
    # ✏️ Return {"name": ..., "weight": ..., "types": [...]}
    #    Remember the User-Agent header (see the pikachu example above).
    #    urllib raises urllib.error.HTTPError for a 404. Catch it, and when
    #    err.code is 404, raise ValueError(f"No Pokémon called {name}") instead.
    pass


bulbasaur = get_pokemon("bulbasaur")
check(bulbasaur["types"], ["grass", "poison"])

try:
    get_pokemon("pikachuuu")
    print("❌ should have raised an error")
except ValueError as err:
    check(str(err), "No Pokémon called pikachuuu")

## 💥 Break it: forgetting `await`

In [ ]:
%%node
// 🔮 My prediction:
function brew(drink) {
  return new Promise((resolve) => setTimeout(() => resolve(`${drink} is ready`), 500));
}

const coffee = brew("Coffee");
console.log(coffee);

In [ ]:
# 🔮 My prediction:
import asyncio

async def brew(drink):
    await asyncio.sleep(0.5)
    return f"{drink} is ready"

coffee = brew("Coffee")
print(coffee)
coffee.close()   # tidy up the coffee we never waited for

🔍 Without `await` you get the **buzzer**, not the coffee: a pending `Promise` in JavaScript, a
`coroutine` object in Python. If you see `Promise { <pending> }` in your output, look for a missing `await`.

## 🌍 Rosetta Stone: same idea, different language
The ideas are the same in every language. Only the spelling changes. Add your own rows when you discover something new.

| Idea | JavaScript | Python |
|---|---|---|
| Run something later | `setTimeout(fn, 1000)` | `time.sleep(1)` (blocks) / `await asyncio.sleep(1)` |
| Async function | `async function f() {}` | `async def f():` |
| Wait for a result | `await f()` | `await f()` |
| Wait for several at once | `await Promise.all([a(), b()])` | `await asyncio.gather(a(), b())` |
| A value that arrives later | `Promise` | coroutine / `Task` |
| Get JSON from a URL | `await (await fetch(url)).json()` | `requests.get(url).json()` |
| Did the request succeed? | `response.ok` | `HTTPError` is raised if not |

## ✍️ Explain it back
Answer in your own words, as if explaining it to a friend who missed today's session.

1. Why doesn't JavaScript just wait (block) while a network request is running?
2. What does `await` do? What happens if you forget it?
3. Why does `Promise.all` make things faster?

**My explanations** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️

## 🌱 Git checkpoint
Work on a branch with a pull request again, but this time ask your teacher to **request changes**.

```bash
git switch -c lesson-11
# ... work and commit ...
git push -u origin lesson-11
```

Open the PR on GitHub. After the review:
1. Read each review comment.
2. Fix the code locally, commit and push. **The PR updates automatically**: no need for a new one.
   ```bash
   git add 11_async_and_fetch.ipynb
   git commit -m "Handle 404 in getPokemon (review feedback)"
   git push
   ```
3. Reply to each comment (for example "Fixed in a1b2c3d") and click **Resolve conversation**.
4. When approved: **Merge**, then `git switch main` and `git pull`.